# Healthcare Analytics: Data Preprocessing
This notebook handles missing values, encodes categorical variables, and performs train-test splitting.


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


## 1. Load Data


In [ ]:
df = pd.read_csv(os.path.join('..', 'data', 'healthcare_dataset.csv'))
print(df.shape)
df.isnull().sum()


## 2. Missing Value Imputation
Since `BMI` has missing values (approx 4%) and is slightly skewed, we will impute with the median value of the training set. To avoid data leakage, we compute the median first.


In [ ]:
median_bmi = df['BMI'].median()
print(f'Median BMI to impute: {median_bmi}')
df['BMI'] = df['BMI'].fillna(median_bmi)
print('Remaining nulls:', df['BMI'].isnull().sum())


## 3. Categorical Variables Encoding
Let's identify categorical variables and encode them appropriately:
- Binary columns (`Gender`, `Ever_Married`, `Residence_Type`): Label Encoding (0 or 1)
- Multi-class columns (`Work_Type`, `Smoking_Status`, `Insurance_Provider`, `Medical_Condition`, `Admission_Type`, `Risk_Category`): One-Hot Encoding


In [ ]:
# Demographics / Predictors we care about for machine learning modeling
feature_cols = ['Age', 'Gender', 'Hypertension', 'Heart_Disease', 'Ever_Married', 
                'Work_Type', 'Residence_Type', 'Avg_Glucose_Level', 'BMI', 'Smoking_Status']
target_col = 'Stroke'

X = df[feature_cols].copy()
y = df[target_col].copy()

# Binary variables label encoding
label_encoders = {}
for col in ['Gender', 'Ever_Married', 'Residence_Type']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le
    print(f'Encoded {col}: {list(le.classes_)} -> {list(le.transform(le.classes_))}')

# One-hot encode remaining multi-class categoricals
X = pd.get_dummies(X, columns=['Work_Type', 'Smoking_Status'], drop_first=True, dtype=int)
print('\nProcessed feature matrix shape:', X.shape)
X.head()


## 4. Train-Test Split
Use stratified splitting because of the target variable's class imbalance.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print('Train class distribution:\n', y_train.value_counts(normalize=True))
print('Test class distribution:\n', y_test.value_counts(normalize=True))


## 5. Save Splits for Next Steps
Save preprocessed train and test sets to temporary CSV files to ensure modularity.


In [ ]:
os.makedirs(os.path.join('..', 'data', 'temp'), exist_ok=True)
X_train.to_csv(os.path.join('..', 'data', 'temp', 'X_train.csv'), index=False)
X_test.to_csv(os.path.join('..', 'data', 'temp', 'X_test.csv'), index=False)
y_train.to_csv(os.path.join('..', 'data', 'temp', 'y_train.csv'), index=False)
y_test.to_csv(os.path.join('..', 'data', 'temp', 'y_test.csv'), index=False)
print('Preprocessed partitions saved to data/temp/')
